# 05 · Export to TensorFlow Lite (CRISP-DM step 6: deployment)

Converts the best model into `kapeklasipika.tflite` (+ `labels.txt`) for the Flutter app,
then checks that the phone model gives the same answers as the original.

In [ ]:
# --- SETUP: run this cell first -----------------------------------------
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Change this if you uploaded the project folder somewhere else in your Drive:
    PROJECT = Path("/content/drive/MyDrive/kapeklasipika")
    %pip install -q imagehash
else:
    # Running on your laptop: walk up from this notebook until we find config/config.yaml
    PROJECT = Path.cwd()
    while not (PROJECT / "config" / "config.yaml").exists():
        PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.config import load_config, use_local_copy
cfg = load_config()
print("Project folder:", PROJECT)

In [ ]:
# Colab only: copy the images to Colab's fast local disk (Drive is slow for 8,000 files)
if IN_COLAB:
    cfg = use_local_copy(cfg)

In [ ]:
import shutil
import numpy as np
import tensorflow as tf
from src.data_loading import load_clean_split
from src.export import to_tflite, write_labels, check_agreement, tflite_predict

MODELS = cfg["paths"]["models"]
best_file = cfg["paths"]["metrics"] / "best_model.txt"            # written by notebook 04
BEST = best_file.read_text().strip() if best_file.exists() else "mobilenetv2_finetuned"
print("Exporting:", BEST)
model = tf.keras.models.load_model(MODELS / f"{BEST}.keras")

tflite_path = MODELS / "kapeklasipika.tflite"
size_mb = to_tflite(model, tflite_path, float16=True)
write_labels(MODELS / "labels.txt")
print(f"Saved {tflite_path.name}: {size_mb} MB (target < 15 MB)")

## Check the phone model against the original
Agreement should be at or very near 100%.

In [ ]:
test_ds = load_clean_split(cfg, "test")
print(f"Keras vs TFLite agreement (first 5 batches): {check_agreement(model, tflite_path, test_ds):.1%}")

# Full test accuracy of the .tflite file itself (this is what runs on the phone)
correct = total = 0
for images, labels in test_ds:
    pred = tflite_predict(tflite_path, images.numpy()).argmax(1)
    correct += int((pred == labels.numpy()).sum()); total += len(pred)
print(f"TFLite test accuracy: {correct / total:.2%} on {total} images")

## Copy the model into the Flutter app

In [ ]:
assets = PROJECT / "07_deployment" / "mobile_app" / "assets"
assets.mkdir(parents=True, exist_ok=True)
shutil.copy2(tflite_path, assets / "kapeklasipika.tflite")
shutil.copy2(MODELS / "labels.txt", assets / "labels.txt")
print("Copied to", assets)